In [ ]:
import numpy as np
from datasets import load_dataset
from PIL import Image
import random
import os

# =============================================================================
# 🧠 코딩 튜터의 필독 설명서 (Learning Corner)
# =============================================================================
# 🚀 데이터셋 개요: naver-clova-ix/cord-v2
# 💡 의미: 이 데이터셋은 '이미지-캡션(Image-Caption)' 데이터셋입니다.
# 🖼️ 목표: 이미지와 그 이미지를 설명하는 문자열(Ground Truth)이 쌍으로 매칭되어 있습니다.
# 🎯 실습 목표: 초보자가 이미지 데이터를 다루는 기초와, 텍스트 요약/분류를 위한 데이터 탐색 능력을 기르는 것입니다.
# 🧑‍🏫 오늘 배울 것: 스트리밍 데이터 로딩, 이미지 객체 처리, 데이터 특성 분석.
# =============================================================================

# 📌 상수 정의 (Constants Setup)
DATASET_NAME = "naver-clova-ix/cord-v2"
SAMPLE_COUNT = 5 # 메모리 효율을 위해 상위 5개 샘플만 사용합니다!

# -----------------------------------------------------------------------------
# 💾 데이터셋 로드 (Data Loading Strategy)
# -----------------------------------------------------------------------------
print("="*80)
print(f"✨ 데이터셋 로딩 시작: {DATASET_NAME}")
print("="*80)

# 1. 스트리밍 모드를 먼저 시도해봅니다. (Fast Check!)
dataset = None
sample_iterator = None

try:
    # 'train' 스플릿을 스트리밍으로 로드 시도 (메모리 효율 최고!)
    print("➡️ [Step 1/3] Streaming 모드 (train split)로 데이터셋 로드 시도...")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 스트리밍 로드 성공! (IterableDataset 타입입니다.)")

except Exception as e:
    # 만약 스트리밍이 안 된다면, 일반 로드 방식을 사용합니다.
    print(f"⚠️ 스트리밍 로드 실패! ({type(e).__name__} 발생). 일반 Dataset으로 전환합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='test') # 테스트 셋의 작은 부분만 로드
    except Exception as e_fall:
        print(f"❌ 데이터셋 로드 실패: {e_fall}")
        exit()


# 2. 스트리밍 여부에 따라 샘플링 패턴을 결정합니다. (매우 중요!)
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋 (IterableDataset)
    print("\n✨ 스트리밍 패턴을 적용하여 샘플링합니다.")
    # take()를 사용하여 지정된 개수만큼의 데이터를 순회할 이터레이터를 만듭니다.
    sample_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)
    print("\n✨ 일반 Dataset 패턴을 적용하여 샘플링합니다.")
    # list(dataset.take(K)) 패턴을 사용합니다.
    dataset = list(dataset.take(SAMPLE_COUNT))
    sample_iterator = dataset


# -----------------------------------------------------------------------------
# 🔍 데이터 탐색 및 실습 (Exploration and Practice)
# -----------------------------------------------------------------------------

# 데이터를 순회하기 위한 리스트 초기화
sample_data_list = []
print("\n" + "#"*80)
print(f"🚀 {SAMPLE_COUNT}개의 샘플 데이터를 순회하며 실습을 시작합니다.")
print("#"*80)

# 데이터셋을 순회하며 샘플을 추출합니다.
# sample_iterator가 iter() 객체인지, 아니면 list 형태인지에 따라 처리합니다.

if isinstance(sample_iterator, type(iter(None))): # 실제로 이터레이터인 경우
    print("🎨 [튜터의 팁] 이터레이터를 순회하는 것은 메모리 절약에 최적입니다!")
    for i, sample in enumerate(sample_iterator):
        sample_data_list.append(sample)
else: # list 형태인 경우 (샘플링 결과가 리스트로 저장된 경우)
    print("💾 [튜터의 팁] 메모리에 작은 샘플 전체를 로드했습니다.")
    sample_data_list = sample_iterator


# =============================================================
# 💡 실습 1: 데이터 구조 이해하기 (Data Structure Understanding)
# =============================================================
print("\n\n=====================================================================")
print("✨ 실습 1: 데이터 구조 및 타입 확인 (What kind of data is this?)")
print("=====================================================================")

print("🎨 첫 번째 샘플을 열어보겠습니다.")

# 첫 번째 샘플에 접근합니다.
sample = sample_data_list[0]

# 1. 'image' (이미지) 확인: 데이터 타입과 크기를 알아봅니다.
image_data = sample['image']
print(f"🖼️ [Image Data] 타입: {type(image_data)}")
if hasattr(image_data, 'size'):
    # PIL Image 객체인 경우 .size 속성을 사용합니다.
    print(f"   -> 크기 확인: {image_data.size} (Width, Height)")
elif hasattr(image_data, 'shape'):
     # Numpy 배열인 경우 .shape 속성을 사용합니다.
    print(f"   -> 크기 확인: {image_data.shape} (Height, Width, Channels)")
else:
    print("   -> 이미지 속성 확인이 어렵습니다.")

# 2. 'ground_truth' (문자열) 확인: 레이블의 유형을 확인합니다.
ground_truth = sample['ground_truth']
print(f"📝 [Ground Truth] 타입: {type(ground_truth)}")
print(f"   -> 값: '{ground_truth[:50]}...' (문자열로 이미지를 묘사합니다)")

# =============================================================
# 🚀 실습 2: 이미지 분석 및 전처리 시뮬레이션 (Image Preprocessing Mock)
# =============================================================
print("\n\n=====================================================================")
print("✨ 실습 2: 이미지 특성 분석 및 전처리 시뮬레이션")
print("=====================================================================")

# 학습자가 데이터 전처리 과정을 이해하도록 돕는 시뮬레이션입니다.
print("🧹 전처리 과정: 모든 이미지를 표준 크기로 리사이징하거나, 텐서로 변환하는 과정이 필요합니다.")
print("   -> 현재는 메모리 절약을 위해 구조만 분석합니다.")

# 첫 번째 샘플을 기준으로 전처리 시뮬레이션을 진행합니다.
for i, sample in enumerate(sample_data_list):
    print(f"\n--- Sample {i+1} 분석 ---")
    image_data = sample['image']
    ground_truth = sample['ground_truth']

    if hasattr(image_data, 'size'):
        # PIL Image 객체를 가정하고 표준 크기 (e.g., 224x224)로 변환하는 과정을 시뮬레이션
        try:
            # 안전하게 임시 이미지 변환 시도 (실제 메모리 부하를 줄이기 위해)
            resized_image = image_data.resize((224, 224))
            print(f"   ✅ 이미지 처리 성공: ({image_data.size[0]}x{image_data.size[1]}) -> 224x224로 스케일 조정 완료!")
        except Exception as e:
            print(f"   ⚠️ 이미지 크기 조정 중 에러 발생 (데이터셋 구조 문제일 수 있습니다): {e}")


# =============================================================
# 🧠 실습 3: 캡셔닝/프롬프트 생성 시뮬레이션 (Creative Application)
# =============================================================
print("\n\n=====================================================================")
print("✨ 실습 3: 창의적 응용 - AI 프롬프트 생성기 역할 놀이")
print("=====================================================================")

print("📝 목표: 이미지(Image)와 설명(Ground Truth)을 조합하여, LLM(거대언어모델)에게 최고의 묘사 프롬프트를 만드는 것을 시뮬레이션합니다.")

for i, sample in enumerate(sample_data_list):
    image_data = sample['image']
    ground_truth = sample['ground_truth']
    
    print(f"\n💡 [Sample {i+1}의 프롬프트 생성]")
    
    # 1. 이미지 메타데이터 (시뮬레이션)
    img_info = ""
    if hasattr(image_data, 'size'):
        img_info = f" (해상도: {image_data.size[0]}x{image_data.size[1]})"
    
    # 2. 프롬프트 조합 (Combining inputs)
    prompt = f"""
    [TASK]: 다음 이미지를 매우 자세하고 생생한 어조로 묘사하는 짧은 문장을 작성해줘.
    [CONTEXT]: 이미지를 묘사한 핵심 키워드는 '{ground_truth}'입니다.
    [IMAGE]: {img_info}
    """
    print("----------------------------------------------------")
    print(f"🤖 완성된 프롬프트:\n{prompt.strip()}")
    print("----------------------------------------------------")
    print("✨ 튜터 코멘트: 'Ground Truth'가 AI에게 추가적인 '힌트'가 되어 프롬프트의 정확도를 높여줍니다!")

print("\n\n=====================================================================")
print("🎉🎉🎉 모든 실습이 완료되었습니다! 🎉🎉🎉")
print("데이터셋의 구조와 데이터가 실제 AI 모델에 어떻게 입력되는지 잘 살펴보셨어요.")
print("지금까지 한 것은 '데이터 탐색(Exploration)'과 '사전 분석(Analysis)' 과정이었다는 것을 기억해 주세요!")
print("👏👏👏 아주 잘하셨어요! 👏👏👏")
print("=====================================================================")